# CyberSentry: An Explainable Phishing and Security Threat Analyzer

**Author:** Anannya_AK  |  **Program:** IBM SkillsBuild Cyber Security 2026 (BharatCares)

CyberSentry reads the **text of a URL** and explains **why** it looks risky or normal. It gives a transparent **0-100 heuristic risk score**, lists every warning sign that fired with its points and a plain-English reason, and suggests what to do next.

> **Important limits**
> * The score is a **heuristic, not a probability**.
> * A **low score does not prove a URL is safe**.
> * A **high score does not prove a URL is malicious**.
> * CyberSentry **never visits** a URL and **never sends** it to any service. It only reads characters.

### How this notebook is organised
| Part | What it contains |
|---|---|
| 1. Settings | Points, keyword lists, risk bands (`config.py` in the project) |
| 2. Validator | Cleans and parses input safely (`validator.py`) |
| 3. Rules | The 16 detection rules (`rules.py`) |
| 4. Analyzer | Adds the points and builds the result (`analyzer.py`) |
| 5. Reports | Table, summary numbers, safe CSV export (`report.py`) |
| 6. Demonstration | Safe made-up URLs, score breakdown, error handling, CSV |
| 7. Limitations | False positives, false negatives, what is not covered |

The interactive **Streamlit dashboard** (`app.py`) is in the GitHub project and cannot run inside a notebook. This notebook contains the full analysis engine that the dashboard uses.

**To run:** `pip install pandas`, then *Run All*. Only `pandas` and the Python standard library (`urllib.parse`, `ipaddress`, `re`) are used.

In [1]:
import ipaddress
import logging
import re
import unicodedata
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime
from typing import Callable, Optional
from urllib.parse import unquote, urlsplit

import pandas as pd

logger = logging.getLogger("cybersentry")
print("Libraries loaded.")

Libraries loaded.


## 1. Settings (all numbers and word lists)

Every point value lives in one place so the scoring is **transparent**: nothing is hidden. The final score is

```
score = min(100, sum of the points of every rule that fired)
```

In [2]:
# ---- Input limits (input is untrusted, so its size is capped) ----
MAX_URL_LENGTH = 2048
MAX_BATCH_URLS = 25

# ---- Risk bands: (lowest score in the band, label). These are LEVELS, not probabilities. ----
RISK_BANDS = [(70, "Critical"), (45, "High"), (20, "Medium"), (0, "Low")]
RISK_LEVEL_ORDER = ["Low", "Medium", "High", "Critical"]

LEVEL_SUMMARY = {
    "Low": "Few or no warning signs were found.",
    "Medium": "Some warning signs were found. Be careful.",
    "High": "Several warning signs were found. Treat this link as suspicious.",
    "Critical": "Many strong warning signs were found. Avoid this link.",
}

LEVEL_ADVICE = {
    "Low": [
        "No strong warning signs were found, but a low score is NOT a guarantee that the link is safe.",
        "Still check who sent the link and whether you were expecting it.",
        "For banking or other sensitive sites, type the address yourself or use a bookmark.",
    ],
    "Medium": [
        "Do not enter passwords or payment details until you have verified the link.",
        "Check with the sender through a different channel (a phone call or the official app).",
        "Open the official website by typing its address yourself instead of using this link.",
    ],
    "High": [
        "Do not click this link or enter any information on the page it leads to.",
        "If it arrived in an email or message, treat that message as suspicious and do not reply.",
        "Report it to your IT/security team or to the service being imitated, if that applies.",
    ],
    "Critical": [
        "Avoid opening this link. Many strong warning signs were found.",
        "If you already entered a password, change it right away on the real site and turn on two-factor authentication.",
        "Report the message to your IT/security team and delete it.",
    ],
}
MAX_RULE_ADVICE = 5

# ---- Points ----
POINTS = {
    "ip_public": 35,                    # public IP address (or a disguised one)
    "ip_private": 20,                   # private / local IP address
    "at_symbol": 25,                    # text before an '@'
    "at_symbol_looks_like_domain": 10,  # ...and it looks like a domain (extra)
    "keyword_each": 5,                  # per phishing-style keyword
    "keyword_max": 15,                  # cap for the keyword rule
    "brand_in_host": 30,                # brand inside someone else's domain
    "brand_real_domain_embedded": 10,   # ...real brand domain used as a decoy (extra)
    "lookalike": 30,                    # paypa1, rnicrosoft
    "punycode_unicode": 25,             # xn-- or non-English letters in the host
    "no_https": 10,                     # plain http://
    "suspicious_tld": 10,               # ending common in phishing reports
    "shortener": 15,                    # link shortener
    "unusual_port": 10,                 # port other than 80 / 443
    "embedded_url": 10,                 # another web address hidden in the URL
    "brand_in_path": 10,                # brand name only in the path/query
    "risky_extension": 10,              # .exe, .scr, .apk ...
}

# Tiered rules: (threshold, points). The FIRST tier that matches is used.
LENGTH_TIERS = [(200, 15), (100, 10), (75, 5)]
SUBDOMAIN_TIERS = [(5, 20), (3, 10)]
HYPHEN_TIERS = [(3, 10), (2, 5)]

# ---- Word lists ----
SUSPICIOUS_KEYWORDS = [
    "login", "log-in", "signin", "sign-in", "verify", "verification",
    "secure", "security", "account", "update", "confirm", "password",
    "bank", "wallet", "billing", "invoice", "payment", "suspend",
    "unlock", "recover", "authenticate", "validate", "webscr",
    "helpdesk", "alert",
]

SUSPICIOUS_TLDS = {
    "tk", "ml", "ga", "cf", "gq", "xyz", "top", "click", "zip", "mov",
    "icu", "cyou", "buzz", "loan", "work", "rest",
}

URL_SHORTENERS = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "is.gd", "ow.ly", "buff.ly",
    "rebrand.ly", "cutt.ly", "shorturl.at", "rb.gy", "tiny.cc", "t.ly",
}

RISKY_EXTENSIONS = (
    ".exe", ".scr", ".bat", ".cmd", ".msi", ".apk", ".jar", ".vbs", ".ps1",
    ".iso", ".dmg", ".docm", ".xlsm", ".zip", ".rar",
)

# For names like example.co.uk the main domain is the last THREE parts.
SECOND_LEVEL_LABELS = {"co", "com", "org", "net", "gov", "edu", "ac"}

# Small DEMO list of brands and their official domains.
BRANDS = {
    "paypal": {"paypal.com", "paypal.me"},
    "google": {"google.com", "google.co.in", "google.co.uk", "googleapis.com", "gstatic.com", "googleusercontent.com"},
    "gmail": {"gmail.com"},
    "microsoft": {"microsoft.com", "microsoftonline.com", "office.com", "office365.com", "live.com", "windows.com", "azure.com"},
    "outlook": {"outlook.com", "live.com", "office.com"},
    "apple": {"apple.com", "icloud.com"},
    "icloud": {"icloud.com", "apple.com"},
    "amazon": {"amazon.com", "amazon.in", "amazon.co.uk", "amazon.de", "amazonaws.com"},
    "facebook": {"facebook.com", "fb.com"},
    "instagram": {"instagram.com"},
    "whatsapp": {"whatsapp.com", "whatsapp.net"},
    "netflix": {"netflix.com"},
    "linkedin": {"linkedin.com", "licdn.com"},
    "dropbox": {"dropbox.com"},
    "github": {"github.com", "github.io", "githubusercontent.com"},
    "adobe": {"adobe.com"},
    "fedex": {"fedex.com"},
    "paytm": {"paytm.com"},
    "phonepe": {"phonepe.com"},
    "flipkart": {"flipkart.com"},
    "hdfcbank": {"hdfcbank.com"},
    "icicibank": {"icicibank.com"},
    "onlinesbi": {"onlinesbi.sbi", "onlinesbi.com"},
    "irctc": {"irctc.co.in"},
}
KNOWN_OFFICIAL_DOMAINS = set().union(*BRANDS.values())

print(f"{len(POINTS)} point settings, {len(BRANDS)} demo brands, {len(SUSPICIOUS_KEYWORDS)} keywords loaded.")

17 point settings, 24 demo brands, 25 keywords loaded.


## 2. Validator: clean, check and parse the input

User input is **untrusted**. The validator:
* rejects empty text, text that is too long, spaces and hidden control characters;
* accepts only `http://` and `https://` (rejects `javascript:`, `ftp://`, ...);
* parses the URL with `urllib.parse.urlsplit` and checks IP addresses with `ipaddress` (no network access);
* **rejects single-word hosts such as `abc` or `localhost`**. This was a bug found during testing: `abc` used to be accepted and scored "Low risk". IP addresses and normal domains still work.

Errors are raised as `InvalidURLError` with a friendly message. The message never repeats the user's text back.

In [3]:
class InvalidURLError(ValueError):
    """Raised when the typed text cannot be treated as a URL we can analyze."""


@dataclass
class ParsedURL:
    """The pieces of a URL that the rules need."""
    original: str
    normalized: str
    scheme: str
    scheme_assumed: bool
    hostname: str
    port: Optional[int]
    userinfo: Optional[str]
    path: str
    query: str
    fragment: str
    labels: list
    ip: Optional[object]
    is_obfuscated_ip: bool
    registered_domain: str
    tld: str
    subdomain_count: int

    @property
    def is_ip_host(self) -> bool:
        return self.ip is not None or self.is_obfuscated_ip


_SCHEME_LIKE = re.compile(r"^([A-Za-z][A-Za-z0-9+.\-]*):(?!\d)")
_HOST_CHARS = re.compile(r"[\w.\-]+")
_NUMERIC_LABEL = re.compile(r"\d+|0x[0-9a-f]+")


def refang(text: str) -> str:
    """Undo 'defanging': hxxps://example[.]com -> https://example.com"""
    text = text.strip()
    text = re.sub(r"^hxxp", "http", text, flags=re.IGNORECASE)
    return text.replace("[://]", "://").replace("[.]", ".")


def defang(text: str) -> str:
    """Make a URL safe to paste in reports: https://example.com -> hxxps://example[.]com"""
    return re.sub(r"http(s?)(?=://)", r"hxxp\1", text, flags=re.IGNORECASE).replace(".", "[.]")


def _registered_domain(labels: list) -> str:
    """Main domain: usually the last two labels, three for names like example.co.uk."""
    if len(labels) >= 3 and len(labels[-1]) == 2 and labels[-2] in SECOND_LEVEL_LABELS:
        return ".".join(labels[-3:])
    return ".".join(labels[-2:])


def parse_url(raw: str) -> ParsedURL:
    """Validate and parse text into a ParsedURL, or raise InvalidURLError."""
    if not isinstance(raw, str):
        raise InvalidURLError("The URL must be text.")

    text = refang(raw)
    if not text:
        raise InvalidURLError("Please enter a URL.")
    if len(text) > MAX_URL_LENGTH:
        raise InvalidURLError(
            f"That text is longer than {MAX_URL_LENGTH} characters, which is more than this tool accepts."
        )
    if any(ch.isspace() or unicodedata.category(ch).startswith("C") for ch in text):
        raise InvalidURLError("The URL contains spaces or hidden control characters. Remove them and try again.")

    # --- scheme handling ---
    if "://" in text:
        scheme = text.split("://", 1)[0].lower()
        if scheme not in ("http", "https"):
            raise InvalidURLError("Only http:// and https:// addresses are supported.")
        normalized, assumed = text, False
    else:
        match = _SCHEME_LIKE.match(text)
        if match:
            raise InvalidURLError(
                f"'{match.group(1)}:' links are not web addresses. "
                "Enter an address that starts with http:// or https://."
            )
        if text.startswith("//"):
            text = text[2:]
        scheme, normalized, assumed = "http", "http://" + text, True

    # --- split into parts (urlsplit never touches the network) ---
    try:
        parts = urlsplit(normalized)
        port = parts.port
        host = parts.hostname
    except ValueError:
        raise InvalidURLError(
            "This does not look like a valid URL. Check the host name and the port number."
        ) from None

    if not host:
        raise InvalidURLError("No host name found. A URL needs a site name, e.g. https://example.com/page")
    if host.endswith("."):
        host = host[:-1]

    netloc = parts.netloc
    userinfo = netloc.rpartition("@")[0] if "@" in netloc else None

    # --- host checks ---
    ip = None
    try:
        ip = ipaddress.ip_address(host)
    except ValueError:
        pass

    if ip is not None:
        labels = [host]
        registered, tld, subdomains, obfuscated = host, "", 0, False
    else:
        if not _HOST_CHARS.fullmatch(host):
            raise InvalidURLError("The host name contains characters that are not allowed in domain names.")
        labels = host.split(".")
        if len(host) > 253 or any(label == "" or len(label) > 63 for label in labels):
            raise InvalidURLError("The host name is not valid (it has empty or overly long parts).")
        obfuscated = all(_NUMERIC_LABEL.fullmatch(label) for label in labels)
        if obfuscated:
            registered, tld, subdomains = host, "", 0
        else:
            if len(labels) == 1:   # fix: single-word hosts such as 'abc' or 'localhost' are rejected
                raise InvalidURLError(
                    "This doesn't look like a valid URL or domain. Try something like example.com."
                )
            registered = _registered_domain(labels)
            tld = labels[-1] if len(labels) > 1 else ""
            subdomains = max(len(labels) - len(registered.split(".")), 0)

    return ParsedURL(
        original=text, normalized=normalized, scheme=scheme, scheme_assumed=assumed,
        hostname=host, port=port, userinfo=userinfo, path=parts.path, query=parts.query,
        fragment=parts.fragment, labels=labels, ip=ip, is_obfuscated_ip=obfuscated,
        registered_domain=registered, tld=tld, subdomain_count=subdomains,
    )


# Quick self-check of the parser
p = parse_url("https://login.example.com:8443/verify?x=1")
print("host:", p.hostname, "| main domain:", p.registered_domain, "| subdomains:", p.subdomain_count, "| port:", p.port)

host: login.example.com | main domain: example.com | subdomains: 1 | port: 8443


## 3. The 16 detection rules

Each rule is **one small function**. It reads a `ParsedURL` and returns either an `Indicator` (what was found, how many points, and a plain-English reason) or `None`. No rule connects to the internet.

A single indicator is never treated as proof of phishing. Many honest sites also use long links, hyphens, `.xyz` domains or words like "login", so most rules add only a few points.

In [4]:
@dataclass
class Indicator:
    """One finding: which rule fired, its points, and why."""
    rule_id: str
    title: str
    points: int
    explanation: str


@dataclass
class Rule:
    """A rule plus the information shown to the user."""
    rule_id: str
    title: str
    max_points: int
    description: str
    advice: str
    check: Callable


def _show(text: str, limit: int = 60) -> str:
    """Shorten long text so explanations stay readable."""
    return text if len(text) <= limit else text[: limit - 1] + "…"


def _is_official(brand: str, registered_domain: str) -> bool:
    return registered_domain in BRANDS[brand]


def _lookalike_variants(hostname: str) -> set:
    """Undo common look-alike tricks: 0->o, 3->e, 4->a, 5->s, 7->t, 1->l or i, rn->m."""
    base = hostname.translate(str.maketrans("03457", "oeast"))
    variants = {base.replace("1", "l"), base.replace("1", "i")}
    variants |= {v.replace("rn", "m") for v in variants}
    return variants


def _decode_idna(hostname: str) -> Optional[str]:
    try:
        return hostname.encode("ascii").decode("idna")
    except (UnicodeError, ValueError):
        return None


def check_ip_host(url):
    host = _show(url.hostname)
    if url.is_obfuscated_ip:
        return Indicator(
            "ip_host", "Number-only host name (disguised IP address)", POINTS["ip_public"],
            f"The host '{host}' is written as plain numbers (decimal, hex or octal). Browsers can turn "
            "this into an IP address. Attackers use this format to hide where a link really goes.",
        )
    if url.ip is None:
        return None
    if url.ip.is_global:
        return Indicator(
            "ip_host", "IP address instead of a domain name", POINTS["ip_public"],
            f"The link goes straight to the IP address {host} instead of a normal domain name. "
            "Real services almost always use a domain name; attackers often use raw IP addresses "
            "because they are cheap, disposable and need no domain registration.",
        )
    return Indicator(
        "ip_host", "IP address in a private/local range", POINTS["ip_private"],
        f"The host {host} is a private, local or reserved address. It only works inside a network "
        "(for example a home or office). Links like this are unusual in emails and messages.",
    )


def check_at_symbol(url):
    if url.userinfo is None:
        return None
    username = url.userinfo.split(":")[0]
    points = POINTS["at_symbol"]
    extra = ""
    if "." in username:
        points += POINTS["at_symbol_looks_like_domain"]
        extra = (" The text before the '@' even looks like a domain name - a classic trick to make "
                 "a link look like it belongs to a trusted site.")
    return Indicator(
        "at_symbol", "'@' symbol in the address", points,
        f"Everything before the '@' ('{_show(username, 40)}') is just a username. The browser really "
        f"connects to '{_show(url.hostname)}'. Attackers put a trusted-looking name in front of the '@' "
        f"to fool people who only read the beginning of the link.{extra}",
    )


def check_url_length(url):
    length = len(url.original)
    for threshold, points in LENGTH_TIERS:
        if length >= threshold:
            return Indicator(
                "url_length", "Unusually long URL", points,
                f"The URL is {length} characters long (the warning starts at {threshold}). Very long "
                "URLs can bury the real domain among confusing text. Note that some honest sites also "
                "produce long links (search results, tracking links).",
            )
    return None


def check_many_subdomains(url):
    if url.is_ip_host:
        return None
    for threshold, points in SUBDOMAIN_TIERS:
        if url.subdomain_count >= threshold:
            return Indicator(
                "many_subdomains", "Excessive subdomains", points,
                f"The host name has {url.subdomain_count} subdomain levels in front of the main domain. "
                "Attackers stack many words at the start so a trusted-looking name appears first. "
                f"Read the host from right to left: the real site is '{_show(url.registered_domain)}'.",
            )
    return None


def check_suspicious_keywords(url):
    text = " ".join([url.hostname, unquote(url.path), unquote(url.query)]).lower()
    found = [word for word in SUSPICIOUS_KEYWORDS if word in text]
    if not found:
        return None
    points = min(len(found) * POINTS["keyword_each"], POINTS["keyword_max"])
    return Indicator(
        "suspicious_keywords", "Phishing-style keywords", points,
        f"Words often used in phishing pages were found: {', '.join(found[:8])}. Phishing pages "
        "pretend to be about logins, accounts or security. Many honest sites use these words too, "
        "so this rule only adds a few points.",
    )


def check_brand_impersonation(url):
    if url.is_ip_host:
        return None
    for brand in BRANDS:
        if brand in url.hostname and not _is_official(brand, url.registered_domain):
            points = POINTS["brand_in_host"]
            extra = ""
            # a decoy is the brand's REAL domain as a whole chunk: "paypal.com.evil.net" (yes), "pineapple.com" (no)
            decoys = [d for d in BRANDS[brand] if re.search(rf"(^|\.){re.escape(d)}(\.|$)", url.hostname)]
            if decoys:
                points += POINTS["brand_real_domain_embedded"]
                extra = (f" It even contains the real address '{decoys[0]}' as a decoy, but that is only "
                         "a subdomain or a piece of someone else's domain.")
            return Indicator(
                "brand_impersonation", "Brand name inside someone else's domain", points,
                f"The host name '{_show(url.hostname)}' contains the brand name '{brand}', but the site's "
                f"real domain is '{_show(url.registered_domain)}', which is not an official {brand} domain "
                f"(according to this tool's small demo list).{extra}",
            )
    return None


def check_lookalike_characters(url):
    if url.is_ip_host:
        return None
    variants = _lookalike_variants(url.hostname)
    for brand in BRANDS:
        if brand in url.hostname or _is_official(brand, url.registered_domain):
            continue   # exact brand names are handled by the brand rule
        if any(brand in variant for variant in variants):
            return Indicator(
                "lookalike_characters", "Look-alike spelling of a brand", POINTS["lookalike"],
                f"The host name '{_show(url.hostname)}' looks like '{brand}' once look-alike characters "
                "are swapped back (for example 0 for o, 1 for l, or 'rn' for 'm'). Attackers register "
                "names that look right at a quick glance.",
            )
    return None


def check_punycode_unicode(url):
    if url.is_ip_host:
        return None
    has_puny = any(label.startswith("xn--") for label in url.labels)
    non_ascii = not url.hostname.isascii()
    if not (has_puny or non_ascii):
        return None
    detail = ""
    if has_puny:
        decoded = _decode_idna(url.hostname)
        if decoded:
            detail = f" The encoded name decodes to '{_show(decoded)}'."
    return Indicator(
        "punycode_unicode", "Internationalized / look-alike letters in the host", POINTS["punycode_unicode"],
        "The host name uses letters from other alphabets (or the 'xn--' encoding for them). Some of these "
        "letters look identical to English letters (for example Cyrillic 'а' and Latin 'a'), which lets "
        f"attackers fake a famous name.{detail} Honest sites in other languages use such names too.",
    )


def check_no_https(url):
    if url.scheme == "http" and not url.scheme_assumed:
        return Indicator(
            "no_https", "Not using HTTPS", POINTS["no_https"],
            "The link uses plain http://, so the connection is not encrypted. Sites that ask for logins "
            "or payments should use https://. (The reverse is NOT true: phishing sites can use https:// too.)",
        )
    return None


def check_many_hyphens(url):
    if url.is_ip_host:
        return None
    count = url.hostname.replace("xn--", "").count("-")
    for threshold, points in HYPHEN_TIERS:
        if count >= threshold:
            return Indicator(
                "many_hyphens", "Many hyphens in the host name", points,
                f"The host name contains {count} hyphens. Phishing domains often chain words together "
                "(like secure-login-bank-example.com) to imitate a trusted name.",
            )
    return None


def check_suspicious_tld(url):
    if url.tld in SUSPICIOUS_TLDS:
        return Indicator(
            "suspicious_tld", "Domain ending often seen in phishing", POINTS["suspicious_tld"],
            f"The domain ends in '.{url.tld}', an ending that appears often in phishing and spam reports "
            "(partly because it is cheap or free to register). Many honest sites use it too, so this is only a mild signal.",
        )
    return None


def check_url_shortener(url):
    if url.registered_domain in URL_SHORTENERS or url.hostname in URL_SHORTENERS:
        return Indicator(
            "url_shortener", "Link-shortening service", POINTS["shortener"],
            f"'{_show(url.registered_domain)}' is a link shortener. It hides the final destination, and "
            "CyberSentry cannot see where it leads because it never opens links.",
        )
    return None


def check_unusual_port(url):
    if url.port not in (None, 80, 443):
        return Indicator(
            "unusual_port", "Unusual port number", POINTS["unusual_port"],
            f"The URL asks for port {url.port}. Ordinary websites use the default ports (80 for http, 443 "
            "for https), which browsers hide. A visible custom port is unusual for a public website.",
        )
    return None


def check_embedded_url(url):
    tail = unquote(unquote(f"{url.path}?{url.query}#{url.fragment}")).lower()   # decode twice: %253A tricks
    if re.search(r"https?://", tail):
        return Indicator(
            "embedded_url", "Another web address hidden inside the URL", POINTS["embedded_url"],
            "A second web address is hidden in the path or query (for example after 'redirect='). "
            "Attackers use this to send you through a trusted-looking page to a harmful one. "
            "Some honest sites (archives, login redirects) do this too.",
        )
    return None


def check_brand_in_path(url):
    if url.registered_domain in KNOWN_OFFICIAL_DOMAINS:
        return None   # e.g. google.com/search?q=paypal is not impersonation
    text = unquote(f"{url.path} {url.query}").lower()
    for brand in BRANDS:
        if brand in text and brand not in url.hostname:
            return Indicator(
                "brand_in_path", "Brand name only in the path", POINTS["brand_in_path"],
                f"The brand name '{brand}' appears after the domain, but the domain "
                f"('{_show(url.registered_domain)}') is not an official {brand} domain. Anyone can put "
                "a brand name in the path of their own site to make a link look official.",
            )
    return None


def check_risky_extension(url):
    path = unquote(url.path).lower()
    for ext in RISKY_EXTENSIONS:
        if path.endswith(ext):
            return Indicator(
                "risky_extension", "Link to a risky file type", POINTS["risky_extension"],
                f"The link points to a file ending in '{ext}'. Programs, scripts, disk images, archives and "
                "macro-enabled documents are common ways to deliver malware. Many honest downloads use these types too.",
            )
    return None


RULES = [
    Rule("ip_host", "IP address instead of a domain name", POINTS["ip_public"],
         "The host is an IP address (or a number disguised as one) instead of a name.",
         "Prefer links that use a proper domain name. If someone told you to visit an IP address, confirm with them through another channel.",
         check_ip_host),
    Rule("at_symbol", "'@' symbol in the address", POINTS["at_symbol"] + POINTS["at_symbol_looks_like_domain"],
         "Text before an '@' in the address, which the browser treats as a username rather than the site.",
         "Read the part AFTER the '@' - that is where the link really goes.",
         check_at_symbol),
    Rule("url_length", "Unusually long URL", LENGTH_TIERS[0][1],
         "URLs of 75, 100 or 200+ characters (more points for longer).",
         "Look at the real domain (the part before the first single '/') instead of the whole long address.",
         check_url_length),
    Rule("many_subdomains", "Excessive subdomains", SUBDOMAIN_TIERS[0][1],
         "Three or more (or five or more) labels in front of the main domain.",
         "Read the host name from right to left: the real domain is the last two parts.",
         check_many_subdomains),
    Rule("suspicious_keywords", "Phishing-style keywords", POINTS["keyword_max"],
         "Words such as login, verify, secure, account or update in the host, path or query (5 points each, capped).",
         "Be careful with pages that ask you to log in, verify or confirm something. Open the official site by typing its address yourself.",
         check_suspicious_keywords),
    Rule("brand_impersonation", "Brand name inside someone else's domain",
         POINTS["brand_in_host"] + POINTS["brand_real_domain_embedded"],
         "A well-known brand name inside a host whose real domain is not the brand's.",
         "Go to the brand's official site by typing its address or using a bookmark rather than this link.",
         check_brand_impersonation),
    Rule("lookalike_characters", "Look-alike spelling of a brand", POINTS["lookalike"],
         "A brand spelled with swapped characters, like paypa1 or rnicrosoft.",
         "Compare the domain with the real brand, letter by letter.",
         check_lookalike_characters),
    Rule("punycode_unicode", "Internationalized / look-alike letters in the host", POINTS["punycode_unicode"],
         "'xn--' encoded labels or non-English letters in the host name (homograph attacks).",
         "Do not enter credentials on domains with unusual characters; type the address yourself.",
         check_punycode_unicode),
    Rule("no_https", "Not using HTTPS", POINTS["no_https"],
         "The address starts with http:// instead of https://.",
         "Avoid entering passwords or payment details on pages without https:// (but https:// alone does not prove a site is safe).",
         check_no_https),
    Rule("many_hyphens", "Many hyphens in the host name", HYPHEN_TIERS[0][1],
         "Two or three (or more) hyphens in the host name.",
         "Be wary of long hyphenated domain names that imitate a brand.",
         check_many_hyphens),
    Rule("suspicious_tld", "Domain ending often seen in phishing", POINTS["suspicious_tld"],
         "A domain ending (like .xyz or .top) that often appears in phishing reports.",
         "Treat links on unfamiliar domain endings with extra care and check where the message really came from.",
         check_suspicious_tld),
    Rule("url_shortener", "Link-shortening service", POINTS["shortener"],
         "A known link shortener such as bit.ly, which hides the real destination.",
         "Do not click shortened links from unknown senders; ask for the full link instead.",
         check_url_shortener),
    Rule("unusual_port", "Unusual port number", POINTS["unusual_port"],
         "A visible port other than 80 or 443.",
         "Avoid links with unusual port numbers unless you know the service.",
         check_unusual_port),
    Rule("embedded_url", "Another web address hidden inside the URL", POINTS["embedded_url"],
         "An http:// or https:// address inside the path or query (possible redirect).",
         "Be careful with links that contain another web address; they may send you somewhere else.",
         check_embedded_url),
    Rule("brand_in_path", "Brand name only in the path", POINTS["brand_in_path"],
         "A brand name after the domain, on a domain that is not the brand's.",
         "Check the domain name, not the page path - anyone can put a brand name in a path.",
         check_brand_in_path),
    Rule("risky_extension", "Link to a risky file type", POINTS["risky_extension"],
         "A link ending in .exe, .scr, .apk, .zip and similar file types.",
         "Do not download or run files from untrusted links; scan them with security software first.",
         check_risky_extension),
]
RULES_BY_ID = {rule.rule_id: rule for rule in RULES}

# Show the rule table (this is the "scoring rulebook")
rulebook = pd.DataFrame(
    [{"Rule": r.title, "Max points": r.max_points, "What it looks for": r.description} for r in RULES]
)
print(f"{len(RULES)} rules loaded.\n")
print(rulebook.to_string(index=False))

16 rules loaded.

                                              Rule  Max points                                                                                          What it looks for
               IP address instead of a domain name          35                                The host is an IP address (or a number disguised as one) instead of a name.
                         '@' symbol in the address          35            Text before an '@' in the address, which the browser treats as a username rather than the site.
                                Unusually long URL          15                                               URLs of 75, 100 or 200+ characters (more points for longer).
                              Excessive subdomains          20                                        Three or more (or five or more) labels in front of the main domain.
                           Phishing-style keywords          15 Words such as login, verify, secure, account or update in the host, p

## 4. Analyzer: run every rule and build the result

```
score = min(100, sum of the points of every rule that fired)
```

| Score | Level |
|---|---|
| 0 - 19 | Low |
| 20 - 44 | Medium |
| 45 - 69 | High |
| 70 - 100 | Critical |

These levels describe a **heuristic score**. They are **not probabilities**.

In [5]:
@dataclass
class AnalysisResult:
    """Everything we learned about one URL."""
    timestamp: str
    input_url: str
    defanged_url: str
    hostname: str
    registered_domain: str
    score: int
    risk_level: str
    indicators: list = field(default_factory=list)
    notes: list = field(default_factory=list)
    recommendations: list = field(default_factory=list)

    def to_row(self) -> dict:
        """Flatten into a plain dict (history table and CSV)."""
        return {
            "timestamp": self.timestamp,
            "url": self.input_url,
            "defanged_url": self.defanged_url,
            "hostname": self.hostname,
            "registered_domain": self.registered_domain,
            "risk_score": self.score,
            "risk_level": self.risk_level,
            "indicator_count": len(self.indicators),
            "indicators": "; ".join(f"{i.title} (+{i.points})" for i in self.indicators),
            "explanations": " | ".join(i.explanation for i in self.indicators),
            "recommendations": " | ".join(self.recommendations),
            "indicator_titles": [i.title for i in self.indicators],
        }


def risk_level_for(score: int) -> str:
    """Convert a 0-100 score into Low / Medium / High / Critical."""
    for minimum, label in RISK_BANDS:
        if score >= minimum:
            return label
    return RISK_BANDS[-1][1]


def analyze_url(raw_url: str, now: Optional[datetime] = None) -> AnalysisResult:
    """Analyze the TEXT of a URL. Raises InvalidURLError if the input is unusable.
    This function never opens the URL or contacts any server."""
    parsed = parse_url(raw_url)
    now = now or datetime.now()

    indicators, notes = [], []
    for rule in RULES:
        try:
            finding = rule.check(parsed)
        except Exception:                  # one buggy rule must not crash the whole analysis
            logger.exception("Rule %s failed", rule.rule_id)
            notes.append(f"The rule '{rule.title}' could not be evaluated for this URL.")
            continue
        if finding is not None:
            indicators.append(finding)

    indicators.sort(key=lambda i: i.points, reverse=True)
    total = sum(i.points for i in indicators)
    score = min(100, total)
    level = risk_level_for(score)

    # Extra information that is shown but NOT scored
    if parsed.scheme_assumed:
        notes.append("No http:// or https:// was typed, so the HTTPS check was skipped.")
    elif parsed.scheme == "https":
        notes.append("The URL uses HTTPS. That only means the connection is encrypted - phishing sites use HTTPS too.")
    if total > 100:
        notes.append(f"The rules added up to {total} points; the score is capped at 100.")
    if not indicators:
        notes.append("No rule matched. That does not prove the URL is safe - these rules cannot see everything.")

    recommendations = list(LEVEL_ADVICE[level])
    for indicator in indicators[:MAX_RULE_ADVICE]:
        advice = RULES_BY_ID[indicator.rule_id].advice
        if advice not in recommendations:
            recommendations.append(advice)

    return AnalysisResult(
        timestamp=now.strftime("%Y-%m-%d %H:%M:%S"),
        input_url=parsed.original,
        defanged_url=defang(parsed.original),
        hostname=parsed.hostname,
        registered_domain=parsed.registered_domain,
        score=score, risk_level=level,
        indicators=indicators, notes=notes, recommendations=recommendations,
    )


print("Analyzer ready. Example level for score 45:", risk_level_for(45))

Analyzer ready. Example level for score 45: High


## 5. Reports: table, summary numbers and safe CSV export

URLs are untrusted input, so the CSV export protects against **CSV (formula) injection**: Excel treats a cell that starts with `=`, `+`, `-` or `@` as a formula, so such cells get an apostrophe in front.

In [6]:
CSV_COLUMNS = [
    "timestamp", "url", "defanged_url", "hostname", "registered_domain",
    "risk_score", "risk_level", "indicator_count", "indicators",
    "explanations", "recommendations",
]


def results_to_dataframe(rows: list) -> pd.DataFrame:
    """rows = list of dicts from AnalysisResult.to_row()."""
    return pd.DataFrame(rows).reindex(columns=CSV_COLUMNS)


def csv_safe(value):
    """Neutralise cells that spreadsheet programs could run as formulas."""
    if isinstance(value, str) and value[:1] in ("=", "+", "-", "@", "\t", "\r"):
        return "'" + value
    return value


def dataframe_to_csv_bytes(df: pd.DataFrame) -> bytes:
    """CSV as UTF-8 bytes with BOM (so Excel on Windows shows accented letters correctly)."""
    safe = df.apply(lambda column: column.map(csv_safe))
    return safe.to_csv(index=False).encode("utf-8-sig")


def summarize(rows: list) -> dict:
    """Numbers for dashboard cards and charts."""
    level_counts = pd.Series(0, index=RISK_LEVEL_ORDER, dtype="int64")
    if not rows:
        return {"total": 0, "average": 0.0, "highest": 0, "high_or_critical": 0,
                "level_counts": level_counts, "top_indicators": pd.Series(dtype="int64")}
    scores = [row["risk_score"] for row in rows]
    for row in rows:
        level_counts[row["risk_level"]] += 1
    titles = Counter(title for row in rows for title in row.get("indicator_titles", []))
    top = pd.Series(dict(titles), dtype="int64").sort_values(ascending=False).head(8)
    return {
        "total": len(rows),
        "average": round(sum(scores) / len(scores), 1),
        "highest": max(scores),
        "high_or_critical": int(level_counts["High"] + level_counts["Critical"]),
        "level_counts": level_counts,
        "top_indicators": top,
    }


print("Report helpers ready.")

Report helpers ready.


## 6. Demonstration

All URLs below are **made-up examples**. They are analysed as text only. **Do not open them.** Links are shown "defanged" (`hxxp://example[.]com`) so nobody clicks them by accident.

### 6.1 Analyze a set of safe test URLs

In [7]:
TEST_URLS = [
    "https://www.example.com/about",
    "http://192.168.1.1/login",
    "http://bit.ly/3AbCdEf",
    "https://www.paypa1-secure.example.com/login",
    "http://45.12.30.77/verify-account/login",
    "http://paypal.com@free-gift.example.xyz/claim",
    "http://secure.login.account.example.com.example.org/verify",
    "http://secure-paypal-login.verify-account.example-site.top/signin?redirect=http://evil.example/steal",
]

history = []          # plays the role of the dashboard's session history
for url in TEST_URLS:
    history.append(analyze_url(url).to_row())

table = results_to_dataframe(history)[["defanged_url", "risk_score", "risk_level", "indicator_count"]]
print(table.to_string(index=False))

                                                                                                defanged_url  risk_score risk_level  indicator_count
                                                                           hxxps://www[.]example[.]com/about           0        Low                0
                                                                              hxxp://192[.]168[.]1[.]1/login          35     Medium                3
                                                                                     hxxp://bit[.]ly/3AbCdEf          25     Medium                2
                                                           hxxps://www[.]paypa1-secure[.]example[.]com/login          40     Medium                2
                                                               hxxp://45[.]12[.]30[.]77/verify-account/login          60       High                3
                                                         hxxp://paypal[.]com@free-gift[.]example[.]xyz/cla

### 6.2 Explain one score, rule by rule

This is the "explainable" part: every point can be traced to a rule and a reason.

In [8]:
def explain(url):
    result = analyze_url(url)
    print("URL (defanged):", result.defanged_url)
    print("-" * 70)
    total = 0
    for ind in result.indicators:
        total += ind.points
        print(f"+{ind.points:>2}  {ind.title}")
        print(f"      Why: {ind.explanation}\n")
    print(f"Sum of points: {total}  ->  Score: {result.score}/100  ->  Level: {result.risk_level}")
    print("\nNotes:")
    for note in result.notes:
        print(" -", note)
    print("\nRecommendations:")
    for tip in result.recommendations:
        print(" -", tip)


explain("http://secure.login.account.example.com.example.org/verify")

URL (defanged): hxxp://secure[.]login[.]account[.]example[.]com[.]example[.]org/verify
----------------------------------------------------------------------
+20  Excessive subdomains
      Why: The host name has 5 subdomain levels in front of the main domain. Attackers stack many words at the start so a trusted-looking name appears first. Read the host from right to left: the real site is 'example.org'.

+15  Phishing-style keywords
      Why: Words often used in phishing pages were found: login, verify, secure, account. Phishing pages pretend to be about logins, accounts or security. Many honest sites use these words too, so this rule only adds a few points.

+10  Not using HTTPS
      Why: The link uses plain http://, so the connection is not encrypted. Sites that ask for logins or payments should use https://. (The reverse is NOT true: phishing sites can use https:// too.)

Sum of points: 45  ->  Score: 45/100  ->  Level: High

Notes:

Recommendations:
 - Do not click this link or 

### 6.3 A harmless URL: a low score is **not** a safety guarantee

In [9]:
explain("https://www.example.com")

URL (defanged): hxxps://www[.]example[.]com
----------------------------------------------------------------------
Sum of points: 0  ->  Score: 0/100  ->  Level: Low

Notes:
 - The URL uses HTTPS. That only means the connection is encrypted - phishing sites use HTTPS too.
 - No rule matched. That does not prove the URL is safe - these rules cannot see everything.

Recommendations:
 - No strong warning signs were found, but a low score is NOT a guarantee that the link is safe.
 - Still check who sent the link and whether you were expecting it.
 - For banking or other sensitive sites, type the address yourself or use a bookmark.


### 6.4 Error handling: invalid, empty and edge-case input

The tool must never crash on bad input. Each case below is rejected with a friendly message that does **not** repeat the user's text.

In [10]:
bad_inputs = {
    "empty text": "",
    "single word 'abc'": "abc",
    "localhost": "http://localhost",
    "javascript: link": "javascript:alert(1)",
    "ftp address": "ftp://example.com/file",
    "contains a space": "http://exa mple.com",
    "too long (3000 characters)": "http://example.com/" + "a" * 3000,
    "bad port number": "http://example.com:99999",
    "not text (None)": None,
}

for label, value in bad_inputs.items():
    try:
        analyze_url(value)
        print(f"{label:<28} -> ACCEPTED (unexpected)")
    except InvalidURLError as err:
        print(f"{label:<28} -> rejected: {err}")

print()
print("Still accepted (valid input):")
for ok in ["example.com", "http://192.168.1.1/login", "HTTPS://WWW.EXAMPLE.COM"]:
    r = analyze_url(ok)
    print(f"  {ok:<28} -> {r.score}/100 {r.risk_level}")

empty text                   -> rejected: Please enter a URL.
single word 'abc'            -> rejected: This doesn't look like a valid URL or domain. Try something like example.com.
localhost                    -> rejected: This doesn't look like a valid URL or domain. Try something like example.com.
javascript: link             -> rejected: 'javascript:' links are not web addresses. Enter an address that starts with http:// or https://.
ftp address                  -> rejected: Only http:// and https:// addresses are supported.
contains a space             -> rejected: The URL contains spaces or hidden control characters. Remove them and try again.
too long (3000 characters)   -> rejected: That text is longer than 2048 characters, which is more than this tool accepts.
bad port number              -> rejected: This does not look like a valid URL. Check the host name and the port number.
not text (None)              -> rejected: The URL must be text.

Still accepted (valid input):
  exa

### 6.5 Summary numbers and risk-level distribution

In [11]:
summary = summarize(history)
print("URLs analysed :", summary["total"])
print("Average score :", summary["average"])
print("Highest score :", summary["highest"])
print("High or Critical:", summary["high_or_critical"])
print("\nRisk-level distribution:")
for level, count in summary["level_counts"].items():
    print(f"  {level:<9} {'#' * int(count):<10} {int(count)}")
print("\nMost common warning signs:")
print(summary["top_indicators"].to_string())

URLs analysed : 8
Average score : 44.4
Highest score : 95
High or Critical: 4

Risk-level distribution:
  Low       #          1
  Medium    ###        3
  High      ###        3
  Critical  #          1

Most common warning signs:
Not using HTTPS                         6
Phishing-style keywords                 5
Domain ending often seen in phishing    2
Link-shortening service                 1
IP address in a private/local range     1
Look-alike spelling of a brand          1
IP address instead of a domain name     1
'@' symbol in the address               1


### 6.6 CSV export (with CSV-injection protection)

In [12]:
df = results_to_dataframe(history)
csv_bytes = dataframe_to_csv_bytes(df)
with open("cybersentry_report.csv", "wb") as f:
    f.write(csv_bytes)
print(f"Saved cybersentry_report.csv ({len(csv_bytes)} bytes, {len(df)} rows)")
print("Columns:", ", ".join(df.columns))

print("\nCSV-injection protection examples:")
for risky in ["=1+1", "+cmd", "@SUM(A1)", "-5", "https://example.com"]:
    print(f"  {risky!r:<24} -> {csv_safe(risky)!r}")

Saved cybersentry_report.csv (11317 bytes, 8 rows)
Columns: timestamp, url, defanged_url, hostname, registered_domain, risk_score, risk_level, indicator_count, indicators, explanations, recommendations

CSV-injection protection examples:
  '=1+1'                   -> "'=1+1"
  '+cmd'                   -> "'+cmd"
  '@SUM(A1)'               -> "'@SUM(A1)"
  '-5'                     -> "'-5"
  'https://example.com'    -> 'https://example.com'


## 7. Limitations (please read)

CyberSentry is a **learning and triage tool**, not a replacement for browser protection, e-mail security gateways, threat-intelligence feeds or human judgment.

**False negatives (a low score on something bad)**
* A brand-new phishing domain with no suspicious words can score 0.
* A hacked legitimate website looks legitimate.
* A shortened link hides its destination, and the tool never opens links.
* Brands outside the small demo list (23 brands) are not recognised.

**False positives (a high score on something harmless)**
* A shop called *pineapple* contains the brand name *apple*.
* Honest sites use long tracking links, redirects, hyphens, `.xyz` domains and international characters.

**Other limits**
* Only the **URL text** is analysed: not the page content, the sender, the domain age, certificates or reputation.
* The "main domain" uses a small built-in list of two-part endings (such as `co.uk`), not the full Public Suffix List.
* Attackers who know these rules can avoid them.
* The score is **not a probability** and has not been scientifically validated.

## 8. Future scope
More brands and keywords, a digit-ratio rule, `data:` URI detection, an optional offline log analyzer, and (only after the rule-based core is solid) machine learning compared against this baseline.

---
*The interactive Streamlit dashboard (`app.py`), 115 automated tests and full documentation are in the GitHub project: https://github.com/Anannyaa97/CyberSentry*